In [1]:
import pandas as pd 

In [5]:
data = pd.read_csv('mock_data.csv')
data.head()

,Mission_id,N_Recommandation,Recommandation,Evaluation,Recommendation_status,Action_id,Action,Échéance,Action_status
0,M0001,R0001,Mettre en place une politique de gestion des a...,Critique,accepté,A0001,Élaborer et diffuser la politique d'accès,2025-06-30,En cours
1,M0001,R0001,Mettre en place une politique de gestion des a...,Critique,accepté,A0002,Former les équipes IT sur la nouvelle politique,2025-07-15,Non échues
2,M0001,R0002,Renforcer les contrôles sur les journaux d'audit,Modéré,rejeté,NaN,NaN,NaN,NaN
3,M0001,R0003,Séparer les environnements de production et de...,Critique,accepté,A0003,Déployer un environnement de staging dédié,2025-08-01,En cours
4,M0001,R0004,Chiffrer les données sensibles en transit,Critique,accepté,A0004,Implémenter TLS 1.3 sur tous les services exposés,2025-09-01,Non échues


In [33]:
data['Échéance']

0     2025-06-30
1     2025-07-15
2            NaN
3     2025-08-01
4     2025-09-01
5     2025-09-01
6     2025-05-31
7     2025-05-20
8     2025-07-01
9            NaN
10    2025-06-15
11    2025-07-30
12    2025-10-01
13    2025-05-20
14           NaN
15    2025-12-01
16    2025-12-01
17    2025-06-01
18    2025-07-01
19           NaN
20    2025-08-15
21    2025-05-20
22    2025-06-30
23    2025-09-01
24           NaN
25    2025-07-31
26    2025-10-01
27    2025-10-01
28           NaN
29    2025-11-01
Name: Échéance, dtype: object

In [34]:
data['Échéance'] = pd.to_datetime(data['Échéance'] ,format="%Y-%m-%d")

In [16]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Mission_id             30 non-null     object
 1   N_Recommandation       30 non-null     object
 2   Recommandation         30 non-null     object
 3   Evaluation             30 non-null     object
 4   Recommendation_status  30 non-null     object
 5   Action_id              24 non-null     object
 6   Action                 24 non-null     object
 7   Échéance               24 non-null     object
 8   Action_status          24 non-null     object
dtypes: object(9)
memory usage: 2.2+ KB


In [39]:
EXPECTED_COLS = {
    'Evaluation':str,
    'Recommendation_status':str,
    'Action_status':str
}
mandatory_cols = EXPECTED_COLS.keys()

for col in mandatory_cols:
    print(col,data[col].unique())

Evaluation ['Critique' 'Modéré' 'Mineur']
Recommendation_status ['accepté' 'rejeté' 'en etude']
Action_status ['En cours' 'Non échues' nan 'Clôturées' 'En continue' 'En retard'
 'Rééchelonnées']


In [50]:
No_NA = ['Mission_id','N_Recommandation','Recommendation_status']

for col in No_NA:
    if data[col].isna().any():
            raise HTTPException(
            status_code=400,
            detail=f" column {col} can't have any None or nan values "
        )

In [49]:
EXPECTED_VALUES = {
    "Evaluation": ["Critique", "Modéré", "Mineur"],
    "Recommendation_status": ["accepté", "rejeté", "en etude"],
    "Action_status": ["En cours", "Non échues", "Clôturées",np.nan, "En continue", "En retard",'Rééchelonnées']
}


for col, allowed in EXPECTED_VALUES.items():
    invalid = ~data[col].isin(allowed)
    if invalid.any():
        raise ValueError(f"Invalid values in {col}: {data[col][invalid].unique().tolist()}")

In [14]:
data['N_Recommandation'].dtypes

dtype('O')

In [89]:
# global KPis 
data[["Recommendation_status","Recommandation"]].groupby(["Recommendation_status"]).nunique().reset_index()

,Recommendation_status,Recommandation
0,accepté,20
1,en etude,3
2,rejeté,3


##### **Recommandations**
- Nombre de recommandations non acceptées
- Nombre de recommandation à l’étude
- Répartition du nombre de recommandations acceptées  par criticité  (critique, modéré, mineur)
##### **Actions**
---
- Nombre total d’actions
- Répartition du nombre d’actions par criticité  (critique, modéré, mineur)
- Répartition par statut (non échues, en cours, clôturées, en continue, en retard, rééchelonné)

- Taux d’acceptation de recommandation = Nombre de recommandations acceptées / Nombre total de recommandations

- Taux de mise en œuvre des actions = Nombre d’actions réalisées / Nombre total d’actions

- Taux de mise en œuvre des actions critiques = Nombre d’actions critiques réalisées / Nombre total d’actions critiques




In [56]:
data.columns

Index(['Mission_id', 'N° Recommandation', 'Recommandation', 'Evaluation',
       'Recommendation_status', 'Action_id', 'Action', 'Échéance',
       'Action_status', 'total_recommendations'],
      dtype='object')

In [ ]:
# Mission 
agg_dict = {
    f"Nb_{status}": (
        "N_Recommandation",
        lambda x, status=status: x[data.loc[x.index, "Recommendation_status"] == status ].nunique()
    )
    for status in data["Recommendation_status"].unique()
}

agg_dict["total_recommendation"] = (
    "N_Recommandation",
    "nunique"
)

results = (
    data.groupby("Mission_id")
    .agg(**agg_dict)
    .reset_index()
)

results

,Mission_id,Nb_accepté,Nb_rejeté,Nb_en etude,total_recommendation
0,M0001,3,1,0,4
1,M0002,3,0,1,4
2,M0003,4,1,0,5
3,M0004,4,0,1,5
4,M0005,6,1,1,8


In [ ]:
# recommendation accepte part evaluation
tableau_2 = (
    data.loc[data['Recommendation_status'] == "accepté",["N_Recommandation","Evaluation"]]
    .groupby("Evaluation")
    .agg(nb_recomendation=("N_Recommandation","nunique"))
    .reset_index()
)
tableau_2

,Evaluation,nb_recomendation
0,Critique,11
1,Mineur,2
2,Modéré,7


In [ ]:
# Repartution action part criticité 
tableau_2 = (
    data[["Action_id","Evaluation"]]
    .groupby("Evaluation")
    .agg(nb_recomendation=("Action_id","nunique"))
    .reset_index()
)
tableau_2

,Evaluation,nb_recomendation
0,Critique,14
1,Mineur,2
2,Modéré,6


In [ ]:
# Repartition action part Mission 
tableau_3 = (
    data[["Action_id","Mission_id"]]
    .groupby("Mission_id")
    .agg(nb_action=("Action_id","nunique"))
    .reset_index()
)
tableau_3

,Mission_id,nb_recomendation
0,M0001,4
1,M0002,4
2,M0003,5
3,M0004,4
4,M0005,6


In [129]:
# Repartition  action part status 
tableau_4 = (
    data[['Action_id',"Action_status"]]
    .groupby("Action_status")
    .agg(nb_action=("Action_id","nunique"))
    .reset_index()
)
tableau_4



,Action_status,nb_action
0,Clôturées,3
1,En continue,2
2,En cours,4
3,En retard,3
4,Non échues,8
5,Rééchelonnées,1


In [95]:
results

,Mission_id,Nb_accepté,Nb_rejeté,Nb_en etude,total_recommendation
0,M0001,3,1,0,4
1,M0002,3,0,1,4
2,M0003,4,1,0,5
3,M0004,4,0,1,5
4,M0005,6,1,1,8


In [182]:
# Taux d’acceptation de recommandation :T1
T1 = round(results['Nb_accepté'].sum() / results['total_recommendation'].sum(),2) * 100
T1.item()

# Taux de mise en œuvre des actions : T2
d = tableau_4.set_index("Action_status")["nb_action"].to_dict()
T2 = round((d["En continue"] + d["Clôturées"]) / tableau_4.iloc[:,1].sum(), 2) * 100
T2.item()

# Taux de mise en œuvre des actions critiques : T3
# ACR : Action Crtitique Realisé
ACR = data[
    (data["Evaluation"] == "Critique") & 
    (
        (data["Action_status"] == "Clôturées") |
        (data["Action_status"] == "En cours")
    )
    ]['Action_id'].nunique()

T3 = round(ACR / tableau_4.iloc[:,1].sum(),3) * 100
T3.item()

28.599999999999998